# SHAP explainers and gradient-boosting trees

## What is SHAP?

In Unit 5, CatBoost gave us a single **feature importance score** per feature — useful, but coarse. It tells us *which* features matter on average, not *how* they push a specific prediction up or down.

**SHAP** (SHapley Additive exPlanations) borrows an idea from cooperative game theory. Imagine the features are players, and the "payout" is the model's prediction for one observation. The Shapley value of a feature is its **fair share of the prediction**, averaged over every possible ordering in which features could be added to the model.

This gives us, for every prediction:

$$f(x) = \underbrace{E[f(X)]}_{\text{base value}} + \sum_i \phi_i(x)$$

where each $\phi_i(x)$ is feature $i$'s SHAP value for that row. We'll verify this **additivity property** explicitly below.

Three plots we'll use today:
- **Bar plot** — mean $|\phi_i|$ across the dataset. Global feature importance.
- **Beeswarm (summary) plot** — every row's SHAP value per feature, colored by the feature's value. Shows direction × magnitude × spread.
- **Waterfall plot** — one row's prediction decomposed additively from the base value.

## TreeExplainer vs Explainer

- **`shap.Explainer(model)`** is a dispatcher — it inspects the model and picks an algorithm (TreeSHAP, LinearSHAP, Kernel/Permutation SHAP, etc.).
- **`shap.TreeExplainer(model)`** goes straight to **Tree SHAP**, which computes *exact* Shapley values in polynomial time by walking the tree structure. It's the right choice for tree ensembles (CatBoost, XGBoost, LightGBM, sklearn forests).

**Rule of thumb:** for tree models, go straight to `TreeExplainer`. It's exact *and* fast. Use `shap.Explainer` when the model isn't a tree.

We'll use the Kaggle [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) dataset — 1460 houses in Ames, Iowa with ~80 features, predicting `SalePrice`.

In [ ]:
# !pip install --upgrade catboost shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Compatibility shim: older catboost calls the deprecated DataFrame.iteritems,
# which pandas 2.0+ removed. Re-add it if missing.
if not hasattr(pd.DataFrame, 'iteritems'):
    pd.DataFrame.iteritems = pd.DataFrame.items

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

---
# Part 1: Numerical features only

Start with just the numeric columns — simplest possible setup.

In [ ]:
df = pd.read_csv("Data/train.csv")
numerical_df = df.select_dtypes(include=['number'])
numerical_df.shape

**Your turn:** define `X` (all numeric columns except `SalePrice`) and `y` (just `SalePrice`). Then do an 80/20 train/test split with `random_state=42`.

In [ ]:
X = # your code here
y = # your code here

X_train, X_test, y_train, y_test = # your code here

**Your turn:** initialize a `CatBoostRegressor` with `iterations=1000`, `learning_rate=0.1`, `depth=6`, `verbose=200`. Fit it on the training set, passing `eval_set=(X_test, y_test)` so you can watch the test loss during training.

In [ ]:
catboost_model = # your code here

# your code here to fit the model

**Your turn:** predict on `X_test` and print the Pearson correlation between `y_test` and the predictions.

In [ ]:
y_pred = # your code here

# your code here to print Pearson r

**Your turn:** use `sns.regplot` to plot predicted vs actual `SalePrice`. Label the axes.

In [ ]:
# your code here

## SHAP on the numerical model

The code below is **demo code** — just run it and focus on interpreting the outputs.

In [ ]:
explainer = shap.TreeExplainer(catboost_model)
shap_values = explainer(X_test)

print("shap_values shape:", shap_values.values.shape)  # (n_test, n_features)
print("base value (expected prediction):", shap_values.base_values[0])

### Bar plot — global importance

Heights = mean $|\phi_i|$ across the test set. This is the SHAP analog of the feature importance you got from CatBoost in Unit 5.

In [ ]:
shap.plots.bar(shap_values, max_display=10)

### Beeswarm plot — direction + spread

Every dot is one row's SHAP value for that feature. Color = the feature's value (red = high, blue = low).

Reading it:
- Features are ordered by global importance (top = most important)
- Dots to the **right** pushed the prediction *up*; dots to the **left** pushed it *down*
- If red dots cluster on the right and blue on the left, the feature has a **positive** relationship with the target (and vice versa)
- Wide spread = the feature matters a lot in some rows, little in others

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

### Waterfall plot — one prediction, decomposed

Pick a single test row. The waterfall starts at the base value (average prediction) and shows how each feature pushes the prediction up (red) or down (blue) until it reaches the final prediction for this row.

In [ ]:
shap.plots.waterfall(shap_values[0], max_display=10)

### Additivity check

SHAP's defining property: **the base value plus all SHAP values equals the model's prediction**, for every row.

In [ ]:
row = 0
reconstructed = shap_values.base_values[row] + shap_values.values[row].sum()
actual        = catboost_model.predict(X_test.iloc[[row]])[0]

print(f"base + sum(phi_i):  {reconstructed:.2f}")
print(f"model prediction:   {actual:.2f}")
print(f"match: {np.isclose(reconstructed, actual)}")

### Dependence (scatter) plot — zoom in on one feature

Pick the top feature and plot SHAP value vs feature value. SHAP will auto-color by the feature it most interacts with.

In [ ]:
top_feature = X_test.columns[np.argmax(np.abs(shap_values.values).mean(axis=0))]
print(f"Top feature: {top_feature}")

shap.plots.scatter(shap_values[:, top_feature], color=shap_values)

---
# Part 2: Add categorical features

Now include the string columns. CatBoost handles them natively — no one-hot encoding. Compare which features rise and fall in importance.

**Your turn:**
1. Reload the CSV
2. Select the object (string) columns into `categorical_df`
3. Fill NaNs in those columns with the string `'NA'` (CatBoost doesn't allow NaN in categoricals)
4. Concatenate categorical + numerical columns into `full_df`
5. Save the list of categorical column names as `categorical_features`

In [ ]:
df = # your code here
categorical_df = # your code here
numerical_df   = # your code here

# your code here: fill NaN in categorical_df with 'NA'

categorical_features = # your code here
full_df = # your code here

print(f"{len(categorical_features)} categorical features")

**Your turn:** split `full_df` into `X` (features) and `y` (`SalePrice`), then 80/20 train/test split with `random_state=42`.

In [ ]:
X = # your code here
y = # your code here

X_train, X_test, y_train, y_test = # your code here

**Your turn:** fit a new `CatBoostRegressor` (same hyperparams as before: `iterations=1000`, `learning_rate=0.1`, `depth=6`, `verbose=200`). **This time pass `cat_features=categorical_features` to `.fit()`** so CatBoost handles them natively.

In [ ]:
catboost_model2 = # your code here

# your code here to fit with cat_features=categorical_features
# and to print Pearson r on the test set

## SHAP on the full model

Now that you've seen the SHAP API in Part 1, implement the same three plots for `catboost_model2`.

**Your turn:**
1. Create a `TreeExplainer` for `catboost_model2` and compute SHAP values on `X_test`
2. Show the bar plot with `max_display=10`
3. Show the beeswarm plot with `max_display=10`
4. Show a waterfall plot for a single row (row 36 is interesting — try others too)

In [ ]:
explainer2 = # your code here
shap_values2 = # your code here

# your code here: bar plot

In [ ]:
# your code here: beeswarm plot

In [ ]:
# your code here: waterfall plot for row 36

---
# Part 3: Your turn

1. Pick a different test row and make a waterfall plot for it. Can you tell a **story** about why the model predicted what it did?
2. Try `shap.plots.scatter(shap_values2[:, 'OverallQual'])` and `shap.plots.scatter(shap_values2[:, 'GrLivArea'])`. How does the SHAP value change with the feature value?
3. Compare the top features between Part 1 (numerical only) and Part 2 (numerical + categorical). Did any new categorical features crack the top 10?

In [ ]:
# 1. Your code here

In [ ]:
# 2. Your code here

In [ ]:
# 3. Your observations here (markdown or print)

---
# Part 4: Try it with your own data

Now apply the full pipeline to a dataset **you** care about — a problem set, a project, a Kaggle CSV, or a file from your computer.

**Your checklist:**
1. Load into a `pandas.DataFrame`
2. Identify the **target** column (must be continuous for regression)
3. Identify **categorical** columns (object / category dtype, or numeric codes that are really categorical)
4. Fill NaNs in categorical columns with a placeholder string
5. Fit a `CatBoostRegressor` with `cat_features=...`
6. Build a `TreeExplainer` and show the bar, beeswarm, and a waterfall plot

The demo below uses seaborn's `diamonds` dataset (price prediction with `cut`, `color`, `clarity` as categoricals). **Replace it with your own data.**

## 4.1 Load and inspect

In [ ]:
# -------- REPLACE THIS BLOCK --------
byod = sns.load_dataset('diamonds')
target_col = 'price'
# -------- END REPLACE --------

print("Shape:", byod.shape)
byod.head()

## 4.2 Prepare features

**Your turn:**
1. Split into `X_byod` (features) and `y_byod` (target)
2. Auto-detect categorical columns with `X_byod.select_dtypes(include=['object', 'category']).columns.tolist()`
3. Fill NaN in those columns with `'MISSING'`
4. Do an 80/20 train/test split (`random_state=42`)

In [ ]:
X_byod = # your code here
y_byod = # your code here

cat_features_byod = # your code here
print("Categorical features:", cat_features_byod)

# your code here: fill NaN in categorical columns with 'MISSING'

X_tr, X_te, y_tr, y_te = # your code here

## 4.3 Fit CatBoost

**Your turn:** fit a `CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, verbose=200)` with `cat_features=cat_features_byod`. Print Pearson r on the test set.

In [ ]:
model_byod = # your code here

# your code here: fit with cat_features=cat_features_byod

# your code here: predict and print Pearson r

## 4.4 Explain with SHAP

**Your turn:** build a `TreeExplainer` on `model_byod`, compute SHAP values on `X_te`, and show the **bar plot**, **beeswarm plot**, and a **waterfall** for any single row.

In [ ]:
explainer_byod = # your code here
shap_values_byod = # your code here

# your code here: bar plot

In [ ]:
# your code here: beeswarm plot

In [ ]:
# your code here: waterfall for any single row

## 4.5 Reflect

- Which features drove predictions on your dataset?
- Were the top SHAP features the ones you *expected* to matter, or were there surprises?
- Look at the beeswarm — did any feature have an unexpected direction (e.g., higher values pushing the prediction *down* when you'd expect the opposite)?

---
# Takeaways

1. **SHAP decomposes each prediction additively** — base value + each feature's contribution sums exactly to the model output. That's what "additive" in SHAP means.
2. **Three plot families cover most of what you need:**
   - `shap.plots.bar` — global ranking (comparable to `get_feature_importance`)
   - `shap.plots.beeswarm` — global ranking + direction + spread
   - `shap.plots.waterfall` — explain a single prediction
3. **Use `TreeExplainer` for tree models** — it's exact and orders of magnitude faster than the generic `shap.Explainer`.
4. **SHAP vs feature importance:** both tell you which features matter globally, but only SHAP tells you *how* they push each individual prediction. That's what makes it usable for case-level explanations (e.g., "why did the model flag this applicant?").